In [ ]:
#@title ① config — EDIT THIS CELL

# --- Path to generated block push data ---
DATA_DIR = "data/raw/block_pushing"   # relative to repo root

# --- Episodes to visualize (list of indices) ---
EPISODES_TO_SHOW = [0, 1, 2]

# --- Video playback FPS ---
DATA_FPS = 10

# --- Display height in pixels ---
DISPLAY_HEIGHT = 400

# --- Gemini API key (for VLM validation, optional) ---
GEMINI_API_KEY = ""   # leave empty to skip VLM cells
GEMINI_MODEL   = "gemini-2.5-pro-preview-05-06"
GEMINI_VIDEO_FPS = 5

In [ ]:
#@title ② imports
import os, glob, json, time
import numpy as np
import cv2
import matplotlib.pyplot as plt
import mediapy as media
from IPython.display import display, Markdown

# Resolve DATA_DIR relative to repo root (works from anywhere)
REPO_ROOT = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath('.'))))
# If running from repo root directly:
if os.path.exists(DATA_DIR):
    ABS_DATA_DIR = DATA_DIR
else:
    ABS_DATA_DIR = os.path.join(REPO_ROOT, DATA_DIR)

episode_files = sorted(glob.glob(os.path.join(ABS_DATA_DIR, "episode_*.npz")))
print(f"Found {len(episode_files)} episodes in {ABS_DATA_DIR}")
if len(episode_files) == 0:
    print("Run generate_data.py first:")
    print("  python soda/option_discovery/supervised/block_pushing/generate_data.py --n-episodes 200")

In [ ]:
#@title ③ helper functions

OPTION_NAMES  = ["reach_first", "push_first", "reach_second", "push_second"]
OPTION_COLORS = {
    "reach_first":  (100, 180, 255),   # light blue
    "push_first":   ( 30,  80, 220),   # dark blue
    "reach_second": (255, 160,  60),   # light orange
    "push_second":  (200,  80,   0),   # dark orange
}
OPTION_ID_TO_NAME = {i: n for i, n in enumerate(OPTION_NAMES)}

def load_episode(idx):
    d = np.load(episode_files[idx])
    return d['images'], d['state'], d['actions'], d['option_id']

def burn_frame_number(frame, idx):
    out = frame.copy()
    h, w = out.shape[:2]
    cv2.putText(out, str(idx), (6, 22),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 1, cv2.LINE_AA)
    return out

def burn_option_overlay(frame, option_name):
    out = frame.copy()
    color = OPTION_COLORS.get(option_name, (255, 255, 255))
    h, w = out.shape[:2]
    cv2.rectangle(out, (0, 0), (w-1, h-1), color, 4)
    cv2.putText(out, option_name, (6, h-10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 1, cv2.LINE_AA)
    return out

def frames_to_mp4(frames, fps, path, option_ids=None):
    h, w = frames[0].shape[:2]
    writer = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
    for i, f in enumerate(frames):
        out = burn_frame_number(f.copy(), i)
        if option_ids is not None and i < len(option_ids):
            out = burn_option_overlay(out, OPTION_ID_TO_NAME[option_ids[i]])
        writer.write(cv2.cvtColor(out, cv2.COLOR_RGB2BGR))
    writer.release()

def show_episode(idx, height=400):
    images, state, actions, option_ids = load_episode(idx)
    n = len(images)
    print(f"Episode {idx}: {n} frames | options: {np.unique(option_ids)}")
    # Count segment transitions
    changes = [i for i in range(1, n) if option_ids[i] != option_ids[i-1]]
    print(f"  {len(changes)+1} segments across {n} frames")
    tmp = f"_ep{idx}_labeled.mp4"
    frames_to_mp4(list(images), fps=DATA_FPS, path=tmp, option_ids=option_ids)
    media.show_video(media.read_video(tmp), height=height)
    os.remove(tmp)
    return images, state, actions, option_ids

print("Helper functions loaded.")

In [ ]:
#@title ④ visualize labeled rollouts
# Shows oracle-labeled option overlays for EPISODES_TO_SHOW.
# Color key:
#   Light blue  = reach_first  (navigating to first block)
#   Dark blue   = push_first   (pushing first block to target)
#   Light orange = reach_second (navigating to second block)
#   Dark orange  = push_second  (pushing second block to target)

cached = {}
for idx in EPISODES_TO_SHOW:
    if idx < len(episode_files):
        print(f"\n--- Episode {idx} ---")
        cached[idx] = show_episode(idx, height=DISPLAY_HEIGHT)
    else:
        print(f"Episode {idx} not available (only {len(episode_files)} episodes generated)")

In [ ]:
#@title ⑤ option distribution across shown episodes

from collections import Counter
fig, axes = plt.subplots(1, len(cached), figsize=(5*len(cached), 3))
if len(cached) == 1: axes = [axes]

for ax, (idx, (images, state, actions, option_ids)) in zip(axes, cached.items()):
    counts = Counter(option_ids)
    names  = OPTION_NAMES
    colors = [tuple(c/255 for c in OPTION_COLORS[n]) for n in names]
    ax.bar(range(len(names)), [counts.get(i, 0) for i in range(len(names))],
           color=colors, edgecolor='black')
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=20, ha='right', fontsize=8)
    ax.set_title(f"Episode {idx}")
    ax.set_ylabel("Frames")

plt.suptitle("Oracle option label distribution")
plt.tight_layout()
plt.show()

In [ ]:
#@title ⑥ (optional) VLM validation — send one episode to Gemini
# Useful to check that a VLM agrees with the oracle labels.
# Skip this cell if you don't have a Gemini API key.

if not GEMINI_API_KEY and not os.environ.get('GEMINI_API_KEY'):
    print("Skipping — set GEMINI_API_KEY in cell ① to run VLM validation.")
else:
    import os
    from google import genai
    from google.genai import types

    os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY or os.environ['GEMINI_API_KEY']
    client = genai.Client()

    VLM_EPISODE = EPISODES_TO_SHOW[0]
    images, _, _, oracle_ids = cached[VLM_EPISODE]

    # Render raw (no oracle overlays) for VLM
    tmp = f"_vlm_ep{VLM_EPISODE}.mp4"
    frames_to_mp4(list(images), fps=DATA_FPS, path=tmp, option_ids=None)

    BLOCK_PUSH_PROMPT = """
    Role: Robotics Data Specialist
    Task: Segment a robot block pushing demonstration into discrete options.

    ## Video Context ##
    A robot arm is pushing two colored blocks to their matching colored target zones on a table.
    The robot pushes one block at a time, in any order.

    **Option Definitions:**
    1. **reach_first**: Robot moving toward the FIRST block it will push. Not yet pushing.
    2. **push_first**: Robot actively pushing the FIRST block toward its target zone.
    3. **reach_second**: Robot moving toward the SECOND block. First block is already at target.
    4. **push_second**: Robot actively pushing the SECOND block toward its target zone.

    Frame numbers are burned in red top-left. Output JSON only after `---`.

    ---
    [{"option": "reach_first", "start": 0, "end": 10}, ...]
    """

    try:
        cf = client.files.upload(file=tmp)
        while cf.state.name == 'PROCESSING':
            time.sleep(3); cf = client.files.get(name=cf.name)
        response = client.models.generate_content(
            model=GEMINI_MODEL,
            contents=types.Content(parts=[
                types.Part(file_data=types.FileData(file_uri=cf.uri, mime_type='video/mp4'),
                           video_metadata=types.VideoMetadata(fps=GEMINI_VIDEO_FPS)),
                types.Part(text=BLOCK_PUSH_PROMPT)
            ]),
            config=types.GenerateContentConfig(temperature=0.0)
        )
        print("VLM response:")
        display(Markdown(response.text))
        print("\nOracle labels:", oracle_ids.tolist())
    finally:
        if 'cf' in locals(): client.files.delete(name=cf.name)
        if os.path.exists(tmp): os.remove(tmp)